# 19% vs 0% errors: investigating a Kubernetes deployment rollout with Elastic Agent Builder

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/kubernetes-deployment-rollout-agent-builder). It reproduces the lab end to
end in about an hour and deletes what it created in the last cell.

Requires Elastic Cloud Hosted 9.5+, Docker, kind, kubectl, Helm, cloudflared, and a `.env` copied from `.env.example`.

In [ ]:
%pip install -q --disable-pip-version-check -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import pathlib
import shutil
import subprocess
import time
import warnings
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch, ElasticsearchWarning

warnings.filterwarnings("ignore", category=ElasticsearchWarning)
load_dotenv(override=True)

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

# Names of everything this notebook creates in Kibana. Clean up deletes exactly these.
NAMESPACE, DEPLOYMENT, CONTAINER = "shop", "orders", "orders"
ESQL_TOOLS = [
    "rollout_compare_replicasets",
    "rollout_history",
    "rollout_errors_by_pod",
    "rollout_error_logs",
    "rollout_replicaset_images",
]
WORKFLOW_TOOL = "rollback_deployment"
BUILTIN_TOOLS = [
    "platform.core.resume_workflow_execution",
    "platform.core.get_workflow_execution_status",
]
AGENT_ID = "rollout-investigator"
WORKFLOW_NAME = "rollback-k8s-deployment"
CONNECTOR_NAME = "Kubernetes API (rollout-lab)"
KIBANA_DIR = pathlib.Path("kibana")

# Kibana API paths in one place.
API = {
    "tools": "/api/agent_builder/tools",
    "agents": "/api/agent_builder/agents",
    "converse": "/api/agent_builder/converse",
    "connectors": "/api/actions/connector",
    "workflow": "/api/workflows/workflow",
    "workflow_delete": "/api/workflows/workflow/{id}?force=true",
    "workflow_run": "/api/workflows/workflow/{id}/run",
    "executions": "/api/workflows/workflow/{id}/executions",
    "execution": "/api/workflows/executions/{id}?includeInput=true&includeOutput=true",
    "resume": "/api/workflows/executions/{id}/resume",
}

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, ok=(200,), **kwargs):
    """Call a Kibana API in the configured space. Raises with the response body on errors."""
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=300, **kwargs
    )
    if response.status_code not in ok and not response.ok:
        raise RuntimeError(
            f"{method} {path} -> {response.status_code}: {response.text[:800]}"
        )
    return response.json() if response.text else {}


def minute_floor(t):
    return t.replace(second=0, microsecond=0)


def minute_ceil(t):
    return minute_floor(t) + timedelta(minutes=1) if (t.second or t.microsecond) else t


def lab_time(key):
    """UTC time that k8s/lab.sh wrote to .env."""
    load_dotenv(override=True)
    return datetime.fromisoformat(os.environ[key].replace("Z", "+00:00"))


def now():
    return datetime.now(timezone.utc)


def wait_until(t, why):
    seconds = (t - now()).total_seconds()
    if seconds > 0:
        print(f"waiting {seconds / 60:.1f} min until {t:%H:%M:%S} UTC ({why})")
        time.sleep(seconds)


def esql(query, start, end, title):
    """Run an ES|QL query over [start, end) and print it as a table. Returns a list of dicts."""
    time_range = {"gte": start.isoformat(), "lt": end.isoformat()}
    response = es.esql.query(query=query, filter={"range": {"@timestamp": time_range}})
    names = [column["name"] for column in response["columns"]]
    print(f"\n{title}: {start:%H:%M}-{end:%H:%M} UTC")
    print(" | ".join(names))
    for row in response["values"]:
        print(" | ".join(str(value) for value in row))
    return [dict(zip(names, row)) for row in response["values"]]


def lab(*args, capture=False):
    """Run k8s/lab.sh, streaming its output into the cell, or return stdout when capture=True."""
    if capture:
        return subprocess.run(
            ["k8s/lab.sh", *args], check=True, text=True, capture_output=True
        ).stdout.strip()
    proc = subprocess.Popen(
        ["k8s/lab.sh", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, ["k8s/lab.sh", *args])


missing = [
    tool
    for tool in ("docker", "kind", "kubectl", "helm", "cloudflared")
    if not shutil.which(tool)
]
assert not missing, f"install first: {missing}"
print(
    f"elasticsearch {es.info()['version']['number']} | kibana {kbn('GET', '/api/status')['version']['number']} | space={SPACE}"
)

## 2. Start the lab: kind, the EDOT Collector and orders 1.4.0

`k8s/lab.sh up`, then 10 minutes of baseline traffic.

In [ ]:
lab("up")
BASELINE_MIN = 10
wait_until(
    lab_time("LAB_START") + timedelta(minutes=BASELINE_MIN), "baseline traffic on 1.4.0"
)

## 3. Release orders 1.5.0 and pause the rollout

Waits 11 minutes with three 1.4.0 pods and two 1.5.0 pods serving traffic.

In [ ]:
lab("release")
RELEASE = lab_time("LAB_RELEASE")
PAUSED = lab_time("LAB_PAUSED")
COHORT_START = minute_ceil(PAUSED)
COHORT_END = COHORT_START + timedelta(minutes=11)
print(
    f"release {RELEASE:%H:%M:%S}, paused {PAUSED:%H:%M:%S}, cohort window {COHORT_START:%H:%M}-{COHORT_END:%H:%M} UTC"
)
wait_until(
    COHORT_END + timedelta(seconds=60),
    "11 minutes of paused rollout, plus one minute of ingest delay",
)
lab("status")

## 4. Line up the Kubernetes rollout with the error rate

In [ ]:
FAILED_PER_MINUTE = """FROM traces-generic.otel-default
| WHERE k8s.namespace.name == "shop" AND k8s.deployment.name == "orders"
    AND kind == "Server" AND span.name != "GET /healthz"
| STATS failed = COUNT(*) WHERE event.outcome == "failure"
    BY minute = BUCKET(@timestamp, 1 minute), service.version"""

SCALING_EVENTS = """FROM logs-k8seventsreceiver.otel-default
| WHERE attributes.k8s.namespace.name == "shop"
    AND k8s.object.kind == "Deployment" AND k8s.object.name == "orders"
    AND k8s.event.reason == "ScalingReplicaSet"
| KEEP @timestamp, body.text
| SORT @timestamp"""

CHART_START = minute_floor(RELEASE) - timedelta(minutes=9)
rows = esql(
    FAILED_PER_MINUTE + "\n| WHERE failed > 0\n| SORT minute, service.version",
    CHART_START,
    minute_floor(now()),
    "failed requests per minute (only minutes with failures)",
)
print("versions with failures:", sorted({row["service.version"] for row in rows}))
events = esql(
    SCALING_EVENTS,
    lab_time("LAB_START") - timedelta(minutes=15),
    now(),
    "ScalingReplicaSet events",
)

## 5. Compare the old and new ReplicaSets side by side

In [ ]:
COMPARE = """FROM traces-generic.otel-default
| WHERE k8s.namespace.name == "shop" AND k8s.deployment.name == "orders"
    AND kind == "Server" AND span.name != "GET /healthz"
| STATS requests = COUNT(*),
        errors = COUNT(*) WHERE event.outcome == "failure",
        p95_ms = PERCENTILE(duration, 95),
        pods = COUNT_DISTINCT(k8s.pod.name)
    BY k8s.replicaset.name, service.version
| EVAL error_pct = ROUND(100.0 * errors / requests, 1), p95_ms = ROUND(p95_ms / 1000000.0, 1)"""

RESOURCES = """TS metrics-kubeletstatsreceiver.otel-default
| WHERE k8s.namespace.name == "shop" AND k8s.deployment.name == "orders"
| STATS cpu_millicores = AVG(AVG_OVER_TIME(k8s.pod.cpu.usage)) * 1000,
        memory_mib = AVG(AVG_OVER_TIME(k8s.pod.memory.working_set)) / 1048576
    BY k8s.replicaset.name
| EVAL cpu_millicores = ROUND(cpu_millicores, 1), memory_mib = ROUND(memory_mib, 1)"""

cohorts = esql(
    COMPARE + "\n| SORT service.version",
    COHORT_START,
    COHORT_END,
    "ReplicaSets side by side",
)
OLD_RS = next(
    row["k8s.replicaset.name"] for row in cohorts if row["service.version"] == "1.4.0"
)
NEW_RS = next(
    row["k8s.replicaset.name"] for row in cohorts if row["service.version"] == "1.5.0"
)
esql(RESOURCES, COHORT_START, COHORT_END, "CPU and memory per ReplicaSet")
print(f"old ReplicaSet {OLD_RS} (1.4.0), new ReplicaSet {NEW_RS} (1.5.0)")

## 6. Find the failing pods of the new ReplicaSet

In [ ]:
BY_POD = """FROM traces-generic.otel-default
| WHERE k8s.namespace.name == "shop" AND k8s.deployment.name == "orders"
    AND kind == "Server" AND span.name != "GET /healthz"
| STATS requests = COUNT(*), errors = COUNT(*) WHERE event.outcome == "failure"
    BY k8s.replicaset.name, k8s.pod.name, k8s.node.name
| EVAL error_pct = ROUND(100.0 * errors / requests, 1)
| SORT k8s.replicaset.name, k8s.pod.name"""

ERROR_LOGS = """FROM logs-generic.otel-default
| WHERE k8s.namespace.name == "shop" AND k8s.deployment.name == "orders"
    AND severity_text == "ERROR"
| GROK body.text "missing field '%{WORD:field}' currency=%{WORD:currency}"
| STATS errors = COUNT(*) BY k8s.replicaset.name, field, currency"""

by_pod = esql(BY_POD, COHORT_START, COHORT_END, "errors per pod and node")
error_logs = esql(
    ERROR_LOGS + "\n| SORT currency",
    minute_floor(RELEASE),
    minute_floor(now()),
    "orders error logs since the release",
)

## 7. Give the rollout investigation to an Agent Builder agent

In [ ]:
def show_turn(reply):
    """Print the tool calls and the final message of one converse turn."""
    for step in reply.get("steps", []):
        if step.get("type") == "tool_call":
            print(f"- tool {step.get('tool_id')} {json.dumps(step.get('params', {}))}")
    print("\n" + reply.get("response", {}).get("message", json.dumps(reply)[:2000]))


def converse(text, conversation_id=None):
    body = {"agent_id": AGENT_ID, "input": text}
    if conversation_id:
        body["conversation_id"] = conversation_id
    reply = kbn("POST", API["converse"], json=body)
    show_turn(reply)
    return reply


kbn(
    "DELETE", f"{API['agents']}/{AGENT_ID}", ok=(200, 404)
)  # an agent that uses the tools blocks their deletion
for tool_id in ESQL_TOOLS:
    kbn("DELETE", f"{API['tools']}/{tool_id}", ok=(200, 404))
    kbn(
        "POST",
        API["tools"],
        json=json.loads((KIBANA_DIR / "tools" / f"{tool_id}.json").read_text()),
    )
    print("tool", tool_id)

agent = json.loads((KIBANA_DIR / "agent.json").read_text())
AGENT_BODY = {
    "id": agent["id"],
    "name": agent["name"],
    "description": agent["description"],
    "access_control": {"access_mode": "public"},
    "configuration": {
        "instructions": agent["instructions"],
        "tools": [{"tool_ids": agent["tools"]}],
    },
}
kbn("POST", API["agents"], json=AGENT_BODY)
print("agent", AGENT_ID, "with", len(agent["tools"]), "tools")

investigation = converse(
    "Errors on the orders Deployment in the shop namespace started climbing after this morning's "
    "release, but not every request fails. Compare the old and new ReplicaSets and tell me which "
    "pods are bad and why."
)
CONVERSATION_ID = investigation.get("conversation_id")

## 8. Build the rollback workflow with an HTTP connector

A cloudflared quick tunnel exposes the kind API server to Kibana, for the lab only.

In [ ]:
lab("rbac")
TOKEN = lab("token", capture=True)
TUNNEL_URL = lab("tunnel-start", capture=True).splitlines()[-1]
print("tunnel", TUNNEL_URL)

# The Kubernetes API answers /version without a token: check the tunnel before using it.
for _ in range(30):
    try:
        if requests.get(f"{TUNNEL_URL}/version", timeout=10).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
print(
    "kubernetes", requests.get(f"{TUNNEL_URL}/version", timeout=10).json()["gitVersion"]
)

for connector in kbn("GET", "/api/actions/connectors"):
    if connector["name"] == CONNECTOR_NAME:
        kbn("DELETE", f"{API['connectors']}/{connector['id']}", ok=(200, 204, 404))
CONNECTOR = kbn(
    "POST",
    API["connectors"],
    json={
        "name": CONNECTOR_NAME,
        "connector_type_id": ".http",
        "config": {"url": TUNNEL_URL, "hasAuth": False, "authType": None},
        "secrets": {"secretHeaders": {"Authorization": f"Bearer {TOKEN}"}},
    },
)
CONNECTOR_ID = CONNECTOR["id"]
print("connector", CONNECTOR_ID)

WORKFLOW_YAML = (
    (KIBANA_DIR / f"{WORKFLOW_NAME}.yaml")
    .read_text()
    .replace("<http-connector-id>", CONNECTOR_ID)
)
WORKFLOW_ID = WORKFLOW_NAME
kbn("DELETE", f"{API['tools']}/{WORKFLOW_TOOL}", ok=(200, 404))
kbn(
    "DELETE", API["workflow_delete"].format(id=WORKFLOW_ID), ok=(200, 204, 404)
)  # hard delete frees the id
created = kbn("POST", API["workflow"], json={"yaml": WORKFLOW_YAML, "id": WORKFLOW_ID})
print("workflow", created.get("id", WORKFLOW_ID), "valid:", created.get("valid"))

kbn(
    "POST",
    API["tools"],
    json={
        "id": WORKFLOW_TOOL,
        "type": "workflow",
        "description": (
            "Start the rollback-k8s-deployment workflow, which rolls a Kubernetes Deployment back to the "
            "image of its last healthy ReplicaSet. Call it only after rollout_compare_replicasets shows the "
            "new ReplicaSet failing and the old one healthy, and only after the user asks for a rollback. "
            "Pass the namespace, deployment, container name, the healthy image (for example orders:1.4.0, "
            "from rollout_replicaset_images) and a one-sentence evidence summary as reason. The workflow "
            "pauses for human approval before it changes anything."
        ),
        "tags": ["kubernetes", "rollout"],
        "configuration": {"workflow_id": WORKFLOW_ID, "wait_for_completion": True},
    },
)

AGENT_BODY["configuration"]["tools"] = [
    {"tool_ids": agent["tools"] + [WORKFLOW_TOOL] + BUILTIN_TOOLS}
]
kbn(
    "PUT",
    f"{API['agents']}/{AGENT_ID}",
    json={k: v for k, v in AGENT_BODY.items() if k != "id"},
)
print("agent tools:", len(AGENT_BODY["configuration"]["tools"][0]["tool_ids"]))

## 9. Approve the Kubernetes rollback in Kibana and verify it

If the agent does not start or resume the workflow, the cell does it through the Workflows API.

In [ ]:
def find_execution(after):
    """Latest execution of the rollback workflow started after `after`."""
    response = kbn("GET", API["executions"].format(id=WORKFLOW_ID))
    executions = (
        response
        if isinstance(response, list)
        else response.get("results", response.get("items", []))
    )
    started = [
        e
        for e in executions
        if e.get("startedAt")
        and datetime.fromisoformat(e["startedAt"].replace("Z", "+00:00")) >= after
    ]
    return max(started, key=lambda e: e["startedAt"]) if started else None


def execution(execution_id):
    return kbn("GET", API["execution"].format(id=execution_id))


def wait_for_status(execution_id, statuses, timeout_s):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        current = execution(execution_id)
        if current.get("status") in statuses:
            return current
        time.sleep(5)
    raise TimeoutError(f"execution {execution_id} still {current.get('status')}")


def step_output(current, step_id):
    for step in current.get("stepExecutions", []):
        if step.get("stepId") == step_id:
            return step.get("output")
    return None


asked = now()
converse("Roll orders back to the last healthy image.", CONVERSATION_ID)
ex = find_execution(asked)
if ex is None:
    print("\nThe agent did not start the workflow: starting it through the API.")
    ex_id = kbn(
        "POST",
        API["workflow_run"].format(id=WORKFLOW_ID),
        json={
            "inputs": {
                "namespace": NAMESPACE,
                "deployment": DEPLOYMENT,
                "container": CONTAINER,
                "good_image": "orders:1.4.0",
                "reason": f"New ReplicaSet {NEW_RS} (1.5.0) fails EUR and GBP orders while {OLD_RS} (1.4.0) has 0% errors.",
            }
        },
    )["workflowExecutionId"]
else:
    ex_id = ex["id"]
EXECUTION_ID = ex_id
pending = wait_for_status(EXECUTION_ID, {"waiting_for_input"}, 180)
print(f"\nexecution {EXECUTION_ID} is {pending['status']}")
print(
    f"approval in Kibana: {KBN_BASE}/app/workflows/{WORKFLOW_ID}?executionId={EXECUTION_ID}"
)

APPROVAL_NOTE = "EUR and GBP orders failing on 1.5.0, fix before the next release."
converse(f"Approved. Roll back to orders:1.4.0. Note: {APPROVAL_NOTE}", CONVERSATION_ID)
time.sleep(10)
# "waiting" is the let_pods_settle wait step; only "waiting_for_input" means no approval yet
if execution(EXECUTION_ID).get("status") == "waiting_for_input":
    print("\nThe agent did not resume the workflow: resuming it through the API.")
    kbn(
        "POST",
        API["resume"].format(id=EXECUTION_ID),
        json={"input": {"approved": True, "notes": APPROVAL_NOTE}},
    )

done = wait_for_status(
    EXECUTION_ID, {"completed", "failed", "cancelled", "timed_out"}, 900
)
print(
    f"\nexecution {done['status']}: {done.get('startedAt')} -> {done.get('finishedAt')}"
)
print(
    "steps:",
    [(s.get("stepId"), s.get("status")) for s in done.get("stepExecutions", [])],
)
print(json.dumps(step_output(done, "report"), indent=2))

In [ ]:
# Timeline: Kubernetes events plus the approval time recorded by the execution.
events = esql(
    SCALING_EVENTS,
    RELEASE - timedelta(minutes=1),
    now(),
    "ScalingReplicaSet events since the release",
)
approve_step = next(
    s for s in done.get("stepExecutions", []) if s.get("stepId") == "approve_rollback"
)
approval = approve_step.get("output") or {}
resumed_at = (done.get("context") or {}).get("resumedAt") or approve_step.get(
    "finishedAt"
)
print("\napproval message:", (approve_step.get("input") or {}).get("message"))
print("approval output:", json.dumps(approval))
print("execution:", done.get("startedAt"), "->", done.get("finishedAt"))
last_bad = [
    e["@timestamp"]
    for e in events
    if NEW_RS in e["body.text"] and e["body.text"].endswith(" to 0")
]
if resumed_at and last_bad:
    RESUMED = datetime.fromisoformat(resumed_at.replace("Z", "+00:00"))
    DRAINED = datetime.fromisoformat(last_bad[-1].replace("Z", "+00:00"))
    print(
        f"approved {RESUMED:%H:%M:%S} UTC, last 1.5.0 pod removed {DRAINED:%H:%M:%S} UTC: "
        f"{(DRAINED - RESUMED).total_seconds():.0f} s after approval"
    )
else:
    DRAINED = now()
    print(
        "could not find the approval time or the final scale-down event; check the execution in Kibana"
    )

verify = step_output(done, "verify_errors") or {}
print("\nverify_errors, last 2 minutes after the rollback:")
print(" | ".join(c["name"] for c in verify.get("columns", [])))
for row in verify.get("values", []):
    print(" | ".join(str(v) for v in row))

In [ ]:
POST_START = minute_ceil(DRAINED) + timedelta(minutes=1)
POST_END = POST_START + timedelta(minutes=14)
wait_until(POST_END + timedelta(seconds=60), "14 minutes of traffic after the rollback")
esql(COMPARE, POST_START, POST_END, "after the rollback")
esql(
    FAILED_PER_MINUTE + "\n| WHERE failed > 0\n| SORT minute, service.version",
    CHART_START,
    POST_END,
    "failed requests per minute, release to end of the lab",
)
lab("status")

## 10. Clean up

In [ ]:
kbn("DELETE", f"{API['agents']}/{AGENT_ID}", ok=(200, 404))
for tool_id in ESQL_TOOLS + [WORKFLOW_TOOL]:
    kbn("DELETE", f"{API['tools']}/{tool_id}", ok=(200, 404))
kbn("DELETE", API["workflow_delete"].format(id=WORKFLOW_NAME), ok=(200, 204, 404))
for connector in kbn("GET", "/api/actions/connectors"):
    if connector["name"] == CONNECTOR_NAME:
        kbn("DELETE", f"{API['connectors']}/{connector['id']}", ok=(200, 204, 404))
lab("down")
print("done")